In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
df = pd.read_csv("extracted_data/sha_disbursements_master_file.csv", index_col=False)

In [3]:
df.shape

(34161, 7)

In [4]:
df.head()

,vendor_name,claims,amount,report_month,report_year,schedule,source_pdf
0,"ST CATHERINE'S ICHUNI MISSION HOSPITAL 4 ,",NaN,786479.99,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf
1,"TEMBEZI MEDICAL SERVICES 1 ,",NaN,859005.76,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf
2,"USHIRIKA DISPENSARY 1 ,",NaN,576842.10,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf
3,"TARACHA HEALTH CENTRE 1 ,",NaN,518997.64,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf
4,"Hunter Medical Center 1 ,",NaN,1810.76,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf


In [5]:
df.describe()

,claims,amount,report_year,schedule
count,0.0,3.416100e+04,34161.0,34161.000000
mean,NaN,1.324319e+06,2025.0,1.183660
std,NaN,5.255641e+07,0.0,0.491548
min,NaN,0.000000e+00,2025.0,1.000000
25%,NaN,2.032000e+04,2025.0,1.000000
50%,NaN,9.000000e+04,2025.0,1.000000
75%,NaN,3.660800e+05,2025.0,1.000000
max,NaN,6.629671e+09,2025.0,4.000000


In [6]:
df = df.drop(columns=["claims"])

In [7]:
df = df[~df['vendor_name'].str.contains(r'page', case=False, na=False)]
df['vendor_name'] = df['vendor_name'].str.lower().str.title()

df = df.sort_values(by="vendor_name").reset_index(drop=True)
df['vendor_name'] = df['vendor_name'].replace(',', 'Unnamed Facility')
df = df[df['vendor_name'].str.strip().str.lower() != 'total']
df = df[df['vendor_name'].str.strip().str.lower() != 'grand total']

In [8]:
# remove a leading number (and surrounding punctuation/spaces) from facility_name
df['vendor_name'] = (
    df['vendor_name']
      .str.replace(r'^\s*[\(\[]?\s*\d+[a-zA-Z]?[\)\].:-]*\s*', '', regex=True)
      .str.replace(r'\s+', ' ', regex=True)
      .str.strip()
)
df['vendor_name'] = df['vendor_name'].str.replace(r'\s*(?:[-–—/()]|\bNo\.?\b|#)*\s*\d+[A-Za-z]?\s*,?\s*$', '', regex=True).str.replace(r'\s{2,}', ' ', regex=True).str.strip().str.rstrip('-–—.,:;()')



In [9]:
df[df['vendor_name'].str.contains(r'\d+\s*,?\s*$', na=False)].shape


(8, 6)

In [10]:
df['year_month'] = pd.to_datetime(
    df['report_year'].astype(str) + '-' + df['report_month'],
    format='%Y-%B'
).dt.strftime('%b-%Y')


In [11]:
df[df['vendor_name'].str.strip().str.lower() == 'grand total']


,vendor_name,amount,report_month,report_year,schedule,source_pdf,year_month


In [12]:
df['vendor_name'] = (
    df['vendor_name'].astype(str).str.normalize("NFKC").str.lower()
      .str.replace(r"[’`]", "'", regex=True)                               # unify apostrophes
      .str.replace(r"\bmurang[\'’` ]?a\b", "murang'a", regex=True)         # muranga/murang a -> murang'a
      .str.replace(r"\brefferal\b", "referral", regex=True)                # fix "refferal"
      .str.replace(r"\bhealthcarelimited\b", "healthcare limited", regex=True)  # split stuck words
      .str.replace(r"\s*-\s*", " - ", regex=True)                          # normalize hyphen spacing
      .str.replace(r"\s+", " ", regex=True).str.strip()
      .str.title()
      .str.replace(r"\bGvs\b", "GVS", regex=True)                          # restore acronyms
      .str.replace("Murang'A", "Murang'a", regex=False)                    # fix title-case artifact
)


In [13]:
df['vendor_name'] = df['vendor_name'].str.replace(r'(?i)\bdispensa[\W_]*r[\W_]*y\b', 'dispensary', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip().str.title()


In [14]:
df['vendor_name'] = df['vendor_name'].str.replace(
    r'(?i)\bhealth service\b', 'health services', regex=True
).str.replace(r'\s+', ' ', regex=True).str.strip().str.title()


In [15]:
df['vendor_name'] = (
    df['vendor_name']
      .str.replace(r'(?i)\bL\s*I\s*M\s*I\s*T\s*E\s*D?\b', 'Limited', regex=True)
      .str.replace(r'\s+', ' ', regex=True)
      .str.strip()
      .str.title()
)


In [16]:
df[df['vendor_name'].str.contains(r'advanced care', case=False, na=False)]["vendor_name"]


1472    Advanced Care Diagnostic And Renal Dialysis Limited
1767    Advanced Care Diagnostic And Renal Dialysis Limited
1768    Advanced Care Diagnostic And Renal Dialysis Limited
3579    Advanced Care Diagnostic And Renal Dialysis Limited
8061    Advanced Care Diagnostic And Renal Dialysis Limited
8062    Advanced Care Diagnostic And Renal Dialysis Limited
Name: vendor_name, dtype: object

In [17]:
df[df["vendor_name"] == "Tumticha Medical Centre Limited"]

,vendor_name,amount,report_month,report_year,schedule,source_pdf,year_month
32929,Tumticha Medical Centre Limited,123200.0,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf,Apr-2025
32930,Tumticha Medical Centre Limited,178722.7,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf,Apr-2025
32931,Tumticha Medical Centre Limited,224000.0,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf,Apr-2025
32932,Tumticha Medical Centre Limited,19680.0,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf,Apr-2025
32933,Tumticha Medical Centre Limited,5600.0,April,2025,1,SHA_PAID_FACILITIES_APRIL_2025.pdf,Apr-2025


In [18]:
df.shape

(34065, 7)

In [19]:
df.head(100)

,vendor_name,amount,report_month,report_year,schedule,source_pdf,year_month
0,,2.085357e+07,July,2025,1,SHA_PAYMENTS_JULY_2025_1.pdf,Jul-2025
1,3Rd Park Hospital Limited,3.360000e+04,July,2025,4,SHA_PAYMENTS_JULY_2025_4.pdf,Jul-2025
2,3Rd Park Hospital Limited,1.310400e+05,August,2025,1,SHA_PAYMENTS_AUGUST_2025_1.pdf,Aug-2025
3,A.C.K. Maseno Hospital,1.895040e+06,July,2025,1,SHA_PAID_FACILITIES_JULY_2025.pdf,Jul-2025
4,A.I.C Githumu Mission Hospital,4.368000e+04,August,2025,2,SHA_PAYMENTS_AUGUST_2025_2.pdf,Aug-2025
5,A.I.C Githumu Mission Hospital,5.370000e+05,July,2025,3,SHA_PAYMENTS_JULY_2025_3.pdf,Jul-2025
6,Kenyatta National Hospital,3.414919e+08,June,2025,2,SHA_PAID_FACILITIES_JUNE_2025-2.pdf,Jun-2025
7,Abidha Health Centre,6.720000e+03,July,2025,1,SHA_PAID_FACILITIES_JULY_2025.pdf,Jul-2025
8,Ack Mt Kenya Hospital Mwea,1.718080e+06,August,2025,1,SHA_PAYMENTS_AUGUST_2025_1.pdf,Aug-2025
9,Afyaline Salgaa Medical Centre,1.344000e+04,August,2025,2,SHA_PAYMENTS_AUGUST_2025_2.pdf,Aug-2025


In [20]:
df.to_csv("extracted_data/sha_disbursements_master.csv", index=False)